# Group 3 Module III Project — Movie Industry Data Analysis

## CRISP-DM: 1. Business Understanding

This project follows the **CRISP-DM (Cross-Industry Standard Process for Data Mining)** framework to guide the analysis from the business problem through to actionable recommendations.

### CRISP-DM Phases Used in This Project
1. **Business Understanding** — define the business problem, objectives and questions.
2. **Data Understanding** — load, inspect and assess the available datasets.
3. **Data Preparation** — clean, transform and prepare the data for analysis.
4. **Modeling / Analytical Methods** — apply grouping, aggregation, ROI analysis and relationship analysis to answer the business questions.
5. **Evaluation** — assess whether the findings answer the original business questions.
6. **Deployment / Recommendations** — translate the findings into practical business actions.

## 1.1 Business Context

We are establishing a new movie studio, but the company has no significant experience in the movie industry. This creates a strategic decision problem: **what type of movies should the new studio produce to maximize its chances of commercial success?**

We use historical movie data to identify patterns in **box-office performance, production investment, release timing, genre and audience ratings**. The goal is to turn the data into insights that support decisions about **what to produce, when to release it and how to manage production investment**.

## 1.2 Business Questions

The analysis is designed to answer three management questions:

1. **Which movie genres should the new studio consider investing in?**
   - Which genres have historically produced strong returns?
   - Which genres demonstrate stronger median ROI?

2. **Which time of the year is best to release movies?**
   - Which release months have the highest average worldwide gross?
   - How does release volume vary by month?

3. **How does production budget relate to movie performance?**
   - Does a higher production budget correspond with higher worldwide gross?
   - Does production budget appear to be related to audience rating?

## CRISP-DM: 2. Data Understanding

### 2.1 Import relevant Libraries & Load the Available Source Data

The project contains movie financial, genre, ratings and review information in CSV/TSV files, together with an IMDb SQLite database.

We first load the available sources so we can understand what information is available for the business questions.

In [3]:
# Import Pandas for data manipulation and analysis.
import pandas as pd

# Import sqlite3 to query the IMDb SQLite database.
import sqlite3

# Import Matplotlib for creating charts.
import matplotlib.pyplot as plt

# Import NumPy for numerical operations and conditional logic.
import numpy as np

# Import Seaborn for statistical plots and regression lines.
import seaborn as sns

In [4]:
# CSV files
bom_movie_gross = pd.read_csv("data/bom.movie_gross.csv/bom.movie_gross.csv")

tmdb_movies = pd.read_csv("data/tmdb.movies.csv/tmdb.movies.csv", index_col=0)

tn_movie_budgets = pd.read_csv("data/tn.movie_budgets.csv/tn.movie_budgets.csv")

# TSV files
rt_movie_info = pd.read_csv(
    "data/rt.movie_info.tsv/rt.movie_info.tsv",
    sep="\t"
)

rt_reviews = pd.read_csv(
    "data/rt.reviews.tsv/rt.reviews.tsv",
    sep="\t", encoding="latin1"
)

### 2.2 Connect to the IMDb SQLite Database

The SQLite database provides structured IMDb information that complements the financial datasets. We establish a connection so relevant tables can be inspected and queried using SQL.

In [5]:
# SQLite database
conn = sqlite3.connect("data/im.db/im.db")

### 2.3 Explore the Available CSV and TSV Datasets

Before cleaning or combining data, we inspect the datasets to understand their **structure, columns, data types, identifiers and sample values**. This helps determine which sources and fields are relevant to the business questions.

#### Explore Movies Earnings 

In [6]:
bom_movie_gross.head()

,title,studio,domestic_gross,foreign_gross,year
0,Toy Story 3,BV,415000000.0,652000000,2010
1,Alice in Wonderland (2010),BV,334200000.0,691300000,2010
2,Harry Potter and the Deathly Hallows Part 1,WB,296000000.0,664300000,2010
3,Inception,WB,292600000.0,535700000,2010
4,Shrek Forever After,P/DW,238700000.0,513900000,2010


#### Explore Movie Genres and Ratings

In [7]:
tmdb_movies.head()

,genre_ids,id,original_language,original_title,popularity,release_date,title,vote_average,vote_count
0,"[12, 14, 10751]",12444,en,Harry Potter and the Deathly Hallows: Part 1,33.533,2010-11-19,Harry Potter and the Deathly Hallows: Part 1,7.7,10788
1,"[14, 12, 16, 10751]",10191,en,How to Train Your Dragon,28.734,2010-03-26,How to Train Your Dragon,7.7,7610
2,"[12, 28, 878]",10138,en,Iron Man 2,28.515,2010-05-07,Iron Man 2,6.8,12368
3,"[16, 35, 10751]",862,en,Toy Story,28.005,1995-11-22,Toy Story,7.9,10174
4,"[28, 878, 12]",27205,en,Inception,27.920,2010-07-16,Inception,8.3,22186


tmdb_movies["original_language"].unique()

In [8]:
tmdb_movies.dtypes

genre_ids                str
id                     int64
original_language        str
original_title           str
popularity           float64
release_date             str
title                    str
vote_average         float64
vote_count             int64
dtype: object

#### Explore Movie Earnings

In [9]:
tn_movie_budgets.head()

,id,release_date,movie,production_budget,domestic_gross,worldwide_gross
0,1,"Dec 18, 2009",Avatar,"$425,000,000","$760,507,625","$2,776,345,279"
1,2,"May 20, 2011",Pirates of the Caribbean: On Stranger Tides,"$410,600,000","$241,063,875","$1,045,663,875"
2,3,"Jun 7, 2019",Dark Phoenix,"$350,000,000","$42,762,350","$149,762,350"
3,4,"May 1, 2015",Avengers: Age of Ultron,"$330,600,000","$459,005,868","$1,403,013,963"
4,5,"Dec 15, 2017",Star Wars Ep. VIII: The Last Jedi,"$317,000,000","$620,181,382","$1,316,721,747"


In [10]:
tn_movie_budgets.shape

(5782, 6)

In [11]:
tn_movie_budgets["id"].nunique()

100

#### Explore Movie Reviews

In [12]:
rt_reviews.head()

,id,review,rating,fresh,critic,top_critic,publisher,date
0,3,A distinctly gallows take on contemporary fina...,3/5,fresh,PJ Nabarro,0,Patrick Nabarro,"November 10, 2018"
1,3,It's an allegory in search of a meaning that n...,NaN,rotten,Annalee Newitz,0,io9.com,"May 23, 2018"
2,3,... life lived in a bubble in financial dealin...,NaN,fresh,Sean Axmaker,0,Stream on Demand,"January 4, 2018"
3,3,Continuing along a line introduced in last yea...,NaN,fresh,Daniel Kasman,0,MUBI,"November 16, 2017"
4,3,... a perverse twist on neorealism...,NaN,fresh,NaN,0,Cinema Scope,"October 12, 2017"


In [13]:
rt_reviews.shape

(54432, 8)

In [14]:
rt_reviews["id"].nunique()

1135

### 2.4 Explore the IMDb Database

The database contains multiple IMDb tables. We inspect the available tables and sample records to identify the fields needed for the analysis.

#### Query and Explore Available Tables

Explore all tables in the imdb database before choosing the tables relevant for the analysis.

In [15]:
tables = pd.read_sql("""
    SELECT name
    FROM sqlite_master
    WHERE type='table';
""", conn)

display(tables)

,name
0,movie_basics
1,directors
2,known_for
3,movie_akas
4,movie_ratings
5,persons
6,principals
7,writers


In [16]:
movie_basics = pd.read_sql(
    "SELECT * FROM movie_basics LIMIT 5",
    conn
)

display(movie_basics)

,movie_id,primary_title,original_title,start_year,runtime_minutes,genres
0,tt0063540,Sunghursh,Sunghursh,2013,175.0,"Action,Crime,Drama"
1,tt0066787,One Day Before the Rainy Season,Ashad Ka Ek Din,2019,114.0,"Biography,Drama"
2,tt0069049,The Other Side of the Wind,The Other Side of the Wind,2018,122.0,Drama
3,tt0069204,Sabse Bada Sukh,Sabse Bada Sukh,2018,NaN,"Comedy,Drama"
4,tt0100275,The Wandering Soap Opera,La Telenovela Errante,2017,80.0,"Comedy,Drama,Fantasy"


In [17]:
directors = pd.read_sql(
    "SELECT * FROM directors LIMIT 5",
    conn
)

display(directors)

,movie_id,person_id
0,tt0285252,nm0899854
1,tt0462036,nm1940585
2,tt0835418,nm0151540
3,tt0835418,nm0151540
4,tt0878654,nm0089502


In [18]:
known_for = pd.read_sql(
    "SELECT * FROM known_for LIMIT 5",
    conn
)

display(known_for)

,person_id,movie_id
0,nm0061671,tt0837562
1,nm0061671,tt2398241
2,nm0061671,tt0844471
3,nm0061671,tt0118553
4,nm0061865,tt0896534


In [19]:
movie_ratings = pd.read_sql(
    "SELECT * FROM movie_ratings LIMIT 5",
    conn
)

display(movie_ratings)

,movie_id,averagerating,numvotes
0,tt10356526,8.3,31
1,tt10384606,8.9,559
2,tt1042974,6.4,20
3,tt1043726,4.2,50352
4,tt1060240,6.5,21


In [20]:
movie_ratings = pd.read_sql(
    "SELECT * FROM movie_ratings LIMIT 5",
    conn
)

display(movie_ratings)

,movie_id,averagerating,numvotes
0,tt10356526,8.3,31
1,tt10384606,8.9,559
2,tt1042974,6.4,20
3,tt1043726,4.2,50352
4,tt1060240,6.5,21


In [21]:
persons = pd.read_sql(
    "SELECT * FROM persons LIMIT 5",
    conn
)

display(persons)

,person_id,primary_name,birth_year,death_year,primary_profession
0,nm0061671,Mary Ellen Bauder,None,None,"miscellaneous,production_manager,producer"
1,nm0061865,Joseph Bauer,None,None,"composer,music_department,sound_department"
2,nm0062070,Bruce Baum,None,None,"miscellaneous,actor,writer"
3,nm0062195,Axel Baumann,None,None,"camera_department,cinematographer,art_department"
4,nm0062798,Pete Baxter,None,None,"production_designer,art_department,set_decorator"


In [22]:
principals = pd.read_sql(
    "SELECT * FROM principals LIMIT 5",
    conn
)

display(principals)

,movie_id,ordering,person_id,category,job,characters
0,tt0111414,1,nm0246005,actor,NaN,"[""The Man""]"
1,tt0111414,2,nm0398271,director,NaN,NaN
2,tt0111414,3,nm3739909,producer,producer,NaN
3,tt0323808,10,nm0059247,editor,NaN,NaN
4,tt0323808,1,nm3579312,actress,NaN,"[""Beth Boothby""]"


In [23]:
writers = pd.read_sql(
    "SELECT * FROM writers LIMIT 5",
    conn
)

display(writers)

,movie_id,person_id
0,tt0285252,nm0899854
1,tt0438973,nm0175726
2,tt0438973,nm1802864
3,tt0462036,nm1940585
4,tt0835418,nm0310087
